# 03 — Preprocessing

Select features, engineer venue-distance features, log-transform the target, and split train/test. **Encoding (OneHot) and scaling (StandardScaler) live inside the sklearn `Pipeline` built in nb04**, not here — that way the saved `rent_pipeline.pkl` can `.predict()` on raw input without the caller having to reproduce a cleaning chain by hand.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

df = pd.read_csv('../data/processed/listings_eda.csv')
print(df.shape)
print(df.columns.tolist())

(2566, 78)
['id', 'price', 'accommodates', 'bedrooms', 'bathrooms', 'beds', 'neighbourhood_cleansed', 'room_type', 'property_type', 'minimum_nights', 'latitude', 'longitude', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'reviews_per_month', 'host_is_superhost', 'host_response_rate', 'host_acceptance_rate', 'host_response_time', 'host_total_listings_count', 'instant_bookable', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'n_amenities', 'amen_wifi', 'amen_kitchen', 'amen_hot_water', 'amen_hair_dryer', 'amen_bed_linens', 'amen_hangers', 'amen_essentials', 'amen_cooking_basics', 'amen_refrigerator', 'amen_dishes_and_silverware', 'amen_iron', 'amen_heating', 'amen_dedicated_workspace', 'amen_self_check_in', 'amen_long_term_stays_allowed', 'amen_oven', 

In [2]:
# Keep only useful features. Amenity + text flags are discovered dynamically by
# `amen_` / `text_` prefix — nb02 generates them, so this scales without manual
# edits. `id` is kept through this notebook so we can join calendar.csv
# aggregates below, then dropped right before saving the train/test splits.
#
# LOW_SHAP_DROP: 27 columns identified as effectively noise by a SHAP analysis on
# the prior-iteration model (mean |SHAP| < 0.003 on the *training* set, computed
# from a 100-sample background + 800-sample explainee — no test-set peeking).
# Mostly near-ubiquitous amenities (wifi 90%, kitchen 90% — present on almost
# every listing so they can't discriminate price) and rarely-mentioned variants
# (freezer, hangers, bathtub); plus the five weakest text keywords; plus
# `instant_bookable` (SHAP #59 in prior model — surprisingly weak); plus
# `review_scores_accuracy` (overlapped with `review_scores_rating`).
LOW_SHAP_DROP = {
    'instant_bookable', 'review_scores_accuracy',
    'text_renovated', 'text_view', 'text_design', 'text_river', 'text_lake',
    'amen_bathtub', 'amen_bed_linens', 'amen_cooking_basics',
    'amen_dedicated_workspace', 'amen_dining_table',
    'amen_dishes_and_silverware', 'amen_elevator', 'amen_freezer',
    'amen_hangers', 'amen_heating', 'amen_kitchen',
    'amen_long_term_stays_allowed', 'amen_oven', 'amen_refrigerator',
    'amen_room_darkening_shades', 'amen_self_check_in', 'amen_stove',
    'amen_washer', 'amen_wifi', 'amen_wine_glasses',
}

AMENITY_FLAGS = [c for c in df.columns
                 if c.startswith('amen_') and c not in LOW_SHAP_DROP]
TEXT_FEATURES = [c for c in df.columns
                 if c.startswith('text_') and c not in LOW_SHAP_DROP]

REVIEW_SCORES = ['review_scores_rating', 'review_scores_cleanliness',
                 'review_scores_checkin', 'review_scores_communication',
                 'review_scores_location', 'review_scores_value']
# review_scores_accuracy dropped — see LOW_SHAP_DROP.

HOST_FEATURES = ['host_is_superhost', 'host_response_rate', 'host_acceptance_rate',
                 'host_response_time', 'host_total_listings_count']

ACTIVITY = ['availability_30', 'availability_60', 'availability_90', 'availability_365',
            'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d',
            'reviews_per_month']

FEATURES = ['id',  # join key only, dropped before train/test split
            'accommodates', 'bedrooms', 'bathrooms', 'beds',
            'neighbourhood_cleansed', 'room_type', 'property_type',
            'minimum_nights', 'latitude', 'longitude',
            'n_amenities'] + ACTIVITY + AMENITY_FLAGS + HOST_FEATURES + REVIEW_SCORES + TEXT_FEATURES
# Note: instant_bookable used to live in the structural block — pruned out by LOW_SHAP_DROP.

TARGET = 'price'

df = df[FEATURES + [TARGET]].copy()
df = df.dropna(subset=[TARGET])
df = df[df[TARGET] > 0]
print(f'After price filter: {df.shape}')
print(f'Pruned {len(LOW_SHAP_DROP)} low-SHAP columns ({len(AMENITY_FLAGS)} amenity + '
      f'{len(TEXT_FEATURES)} text flags kept).')

After price filter: (2566, 51)
Pruned 27 low-SHAP columns (10 amenity + 9 text flags kept).


## Event venue proximity features
Distance (km) from each listing to major Zurich event venues.
These vary per listing via lat/lng and act as a proxy for event-driven demand.

In [3]:
# Haversine distance (km) — no external library needed
def haversine_km(lat, lon, vlat, vlon):
    R = 6371.0
    lat, lon, vlat, vlon = map(np.radians, [lat, lon, vlat, vlon])
    a = np.sin((vlat-lat)/2)**2 + np.cos(lat)*np.cos(vlat)*np.sin((vlon-lon)/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Major Zurich event venues (lat, lon)
VENUES = {
    "dist_hallenstadion_km": (47.4317, 8.5495),   # indoor arena
    "dist_letzigrund_km":    (47.3794, 8.5033),   # football stadium
    "dist_messe_km":         (47.4100, 8.5486),   # exhibition centre
    "dist_opernhaus_km":     (47.3654, 8.5464),   # opera house
    "dist_hb_km":            (47.3779, 8.5403),   # main station / event hub
}

for col, (vlat, vlon) in VENUES.items():
    df[col] = haversine_km(df["latitude"].values, df["longitude"].values, vlat, vlon)

df["min_dist_venue_km"] = df[[c for c in VENUES]].min(axis=1)
print("Venue distance features added:")
print(df[[*VENUES, "min_dist_venue_km"]].describe().round(2))

Venue distance features added:
       dist_hallenstadion_km  dist_letzigrund_km  dist_messe_km  \
count                2566.00             2566.00        2566.00   
mean                    6.27                3.11           4.20   
std                     2.01                1.58           1.73   
min                     0.47                0.11           0.11   
25%                     5.32                1.89           3.16   
50%                     6.60                2.98           4.48   
75%                     7.64                4.40           5.38   
max                    12.24                8.16           9.88   

       dist_opernhaus_km  dist_hb_km  min_dist_venue_km  
count            2566.00     2566.00            2566.00  
mean                2.77        2.39               1.30  
std                 1.66        1.33               0.82  
min                 0.18        0.08               0.08  
25%                 1.49        1.36               0.72  
50%              

## Calendar-derived features
Per-listing aggregates from `data/raw/calendar.csv.gz` (the daily availability/min-nights
table that ships with the same Inside Airbnb snapshot). The calendar's `price` column
is fully NaN in recent snapshots, so we cannot use price-based aggregates and would
not want to anyway (target leakage). Instead we extract booking-strategy and event-period
signals that are genuinely orthogonal to anything in `listings.csv`:

- `cal_min_nights_median` / `cal_min_nights_std` — typical min-stay setting and its
  variation across the year (high std = host changes min-nights by season → pricing
  strategy indicator).
- `cal_max_nights_median` — typical max-stay setting.
- `cal_avail_weekend_premium` — (weekday availability rate) − (weekend availability
  rate). Positive = weekends booked harder than weekdays (typical tourist listing);
  near zero = business-travel listing or constant occupancy.
- `cal_avail_rate_zff_2025` — fraction of Sep 29 – Oct 5 2025 available (the snapshot
  was scraped Sep 29, ZFF runs Sep 25 – Oct 5; this is a direct event-pressure signal).
- `cal_avail_rate_winter_hols` — Dec 20 2025 – Jan 2 2026 availability rate (Xmas
  period demand).

In [4]:
cal = pd.read_csv('../data/raw/calendar.csv.gz', parse_dates=['date'])
cal['avail']      = (cal['available'] == 't').astype(int)
cal['is_weekend'] = cal['date'].dt.dayofweek >= 5
cal['is_zff']     = (cal['date'] >= '2025-09-29') & (cal['date'] <= '2025-10-05')
cal['is_xmas']    = (cal['date'] >= '2025-12-20') & (cal['date'] <= '2026-01-02')

cal_agg = cal.groupby('listing_id').agg(
    cal_min_nights_median=('minimum_nights', 'median'),
    cal_min_nights_std=   ('minimum_nights', 'std'),
    cal_max_nights_median=('maximum_nights', 'median'),
)
# Weekend premium and event-period rates need conditional groupbys.
weekday_av = cal[~cal['is_weekend']].groupby('listing_id')['avail'].mean()
weekend_av = cal[ cal['is_weekend']].groupby('listing_id')['avail'].mean()
cal_agg['cal_avail_weekend_premium'] = weekday_av - weekend_av
cal_agg['cal_avail_rate_zff_2025']   = cal[cal['is_zff']].groupby('listing_id')['avail'].mean()
cal_agg['cal_avail_rate_winter_hols'] = cal[cal['is_xmas']].groupby('listing_id')['avail'].mean()

# Left-join: listings without calendar coverage (shouldn't happen for this snapshot
# but defensive) get NaN, handled by SimpleImputer in nb04.
df = df.merge(cal_agg, left_on='id', right_index=True, how='left')

print(f'Calendar features merged. Listings missing calendar data: '
      f'{df["cal_min_nights_median"].isna().sum()}')
print('\nCalendar feature stats:')
print(df[cal_agg.columns.tolist()].describe().round(3))

Calendar features merged. Listings missing calendar data: 0

Calendar feature stats:
       cal_min_nights_median  cal_min_nights_std  cal_max_nights_median  \
count               2566.000            2566.000               2566.000   
mean                  10.945               6.100                558.553   
std                   21.607              13.808                449.722   
min                    1.000               0.000                  1.000   
25%                    1.000               0.000                 90.000   
50%                    3.000               0.000                365.000   
75%                   10.000               0.694               1125.000   
max                  360.000              65.292               1125.000   

       cal_avail_weekend_premium  cal_avail_rate_zff_2025  \
count                   2566.000                 2566.000   
mean                      -0.002                    0.297   
std                        0.023                    0.35

In [5]:
# NA handling and encoding are now done inside the sklearn Pipeline in nb04
# (SimpleImputer + OneHotEncoder). We just write the raw, engineered features here.
print(f'Rows after feature selection + price filter: {len(df)}')
print(f'NA counts (handled by SimpleImputer in nb04):')
print(df.isna().sum()[df.isna().sum() > 0])

Rows after feature selection + price filter: 2566
NA counts (handled by SimpleImputer in nb04):
bedrooms                         7
bathrooms                        9
beds                             1
reviews_per_month              614
host_response_rate             213
host_acceptance_rate           174
host_response_time             213
review_scores_rating           614
review_scores_cleanliness      614
review_scores_checkin          614
review_scores_communication    614
review_scores_location         614
review_scores_value            614
dtype: int64


In [6]:
X = df.drop(columns=[TARGET, 'id'])  # 'id' was only kept for the calendar join
y = np.log1p(df[TARGET])  # log-transform: stabilises right-skewed price distribution

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'\nColumn dtypes in X_train:')
print(X_train.dtypes.value_counts())

Train: (2052, 61), Test: (514, 61)

Column dtypes in X_train:
int64      31
float64    27
object      3
Name: count, dtype: int64


In [7]:
os.makedirs('../data/processed', exist_ok=True)
# Saved with categoricals as plain strings — the Pipeline in nb04 handles encoding.
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)
print('Splits saved (raw features, no encoding / no scaling).')

Splits saved (raw features, no encoding / no scaling).
